# Remaining Experiments (3-5) + YOLO26 Comparison
**Dissertation:** Improving Detection of Visually Challenging Automotive Components

**Prerequisites:**
- Experiments 1 & 2 already complete
- Dataset already downloaded
- Results saving to Google Drive

**Settings:** epochs=50, patience=10 (faster training, same quality)

**IMPORTANT:** Go to Runtime > Change runtime type > T4 GPU

## 0. Setup (run every session)

In [ ]:
# Install and verify GPU
!pip install ultralytics roboflow -q

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT = '/content/drive/MyDrive/dissertation_results'
os.makedirs(DRIVE_PROJECT, exist_ok=True)
print(f"Results folder: {DRIVE_PROJECT}")

In [ ]:
# Download dataset (if not already cached)
import os

if os.path.exists('/content/dataset/data.yaml'):
    print("Dataset already exists, skipping download.")
else:
    from roboflow import Roboflow
    ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"  # <-- REPLACE THIS
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace("team-data").project("car-parts-ybiev")
    version = project.version(1)
    dataset = version.download("yolov8", location="/content/dataset")
    print("Dataset downloaded!")

In [ ]:
# Common imports and settings
from ultralytics import YOLO
import json
import time
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

# Load class names
with open('/content/dataset/data.yaml', 'r') as f:
    data_config = yaml.safe_load(f)
CLASS_NAMES = data_config['names']

# Hard classes identified from Experiment 1
HARD_CLASSES = ['IGNITION COIL', 'GAS CAP', 'DISTRIBUTOR', 'OVERFLOW TANK', 'OIL PRESSURE SENSOR']

# Common training settings (faster than 100 epochs)
COMMON_ARGS = {
    'data': '/content/dataset/data.yaml',
    'epochs': 50,
    'patience': 10,
    'device': 0,
    'save': True,
    'plots': True,
}

def extract_results(val_results, class_names):
    """Extract per-class results from validation."""
    results = {
        'overall': {
            'mAP50': round(float(val_results.box.map50), 4),
            'mAP50_95': round(float(val_results.box.map), 4),
            'precision': round(float(val_results.box.mp), 4),
            'recall': round(float(val_results.box.mr), 4),
        },
        'per_class': {}
    }
    for i, name in enumerate(class_names):
        if i < len(val_results.box.ap50):
            results['per_class'][name] = {
                'mAP50': round(float(val_results.box.ap50[i]), 4),
                'precision': round(float(val_results.box.p[i]), 4) if i < len(val_results.box.p) else None,
                'recall': round(float(val_results.box.r[i]), 4) if i < len(val_results.box.r) else None,
            }
    return results

def print_hard_classes(results, experiment_name):
    """Print results for the 5 hard classes."""
    print(f"\n{'='*60}")
    print(f"{experiment_name} — HARD CLASSES PERFORMANCE")
    print(f"{'='*60}")
    print(f"Overall mAP@0.5: {results['overall']['mAP50']}")
    print(f"\n{'Class':<25} {'mAP50':>8} {'Precision':>10} {'Recall':>8}")
    print('-' * 55)
    for cls in HARD_CLASSES:
        if cls in results['per_class']:
            d = results['per_class'][cls]
            print(f"{cls:<25} {d['mAP50']:>8} {d['precision']:>10} {d['recall']:>8}")

print("Setup complete! Hard classes:", HARD_CLASSES)

---
## Experiment 3: Resolution Impact (RQ2)
**Question:** Does higher input resolution improve detection of visually challenging components?

**Setup:** YOLOv8s at 320px, 640px, and 800px

**Time estimate:** ~2.5 hours total (3 runs x ~50 min each)

In [ ]:
resolutions = [320, 640, 800]
exp3_results = {}

for imgsz in resolutions:
    print(f"\n{'='*60}")
    print(f"Training YOLOv8s at {imgsz}px resolution")
    print(f"{'='*60}")

    model = YOLO('yolov8s.pt')
    start = time.time()

    model.train(
        **COMMON_ARGS,
        imgsz=imgsz,
        batch=16 if imgsz <= 640 else 8,
        project=f'{DRIVE_PROJECT}/runs',
        name=f'exp3_res{imgsz}',
    )

    train_time = time.time() - start

    # Evaluate on test set
    best = YOLO(f'{DRIVE_PROJECT}/runs/exp3_res{imgsz}/weights/best.pt')
    val = best.val(data='/content/dataset/data.yaml', split='test', imgsz=imgsz)

    exp3_results[str(imgsz)] = extract_results(val, CLASS_NAMES)
    exp3_results[str(imgsz)]['training_time_min'] = round(train_time / 60, 1)

    print_hard_classes(exp3_results[str(imgsz)], f"Resolution {imgsz}px")
    print(f"Training time: {train_time/60:.1f} min")

# Save results
with open(f'{DRIVE_PROJECT}/exp3_resolution.json', 'w') as f:
    json.dump(exp3_results, f, indent=2)

print("\n\nEXPERIMENT 3 COMPLETE — Results saved!")
print("\nResolution comparison (overall mAP@0.5):")
for res, data in exp3_results.items():
    print(f"  {res}px: {data['overall']['mAP50']}")

---
## Experiment 4: Augmentation Strategies (RQ3)
**Question:** Which augmentation techniques most improve detection of hard-to-detect parts?

**Setup:** Three augmentation configs — none, standard, advanced (mosaic+mixup)

**Time estimate:** ~2.5 hours total (3 runs x ~50 min each)

In [ ]:
aug_configs = {
    'no_aug': {
        'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0,
        'degrees': 0.0, 'translate': 0.0, 'scale': 0.0,
        'fliplr': 0.0, 'mosaic': 0.0, 'mixup': 0.0,
    },
    'standard_aug': {
        'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4,
        'degrees': 10.0, 'translate': 0.1, 'scale': 0.5,
        'fliplr': 0.5, 'mosaic': 0.0, 'mixup': 0.0,
    },
    'advanced_aug': {
        'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4,
        'degrees': 15.0, 'translate': 0.1, 'scale': 0.5,
        'fliplr': 0.5, 'mosaic': 1.0, 'mixup': 0.15,
    },
}

exp4_results = {}

for name, aug_params in aug_configs.items():
    print(f"\n{'='*60}")
    print(f"Training with: {name}")
    print(f"{'='*60}")

    model = YOLO('yolov8s.pt')
    start = time.time()

    model.train(
        **COMMON_ARGS,
        imgsz=640,
        batch=16,
        project=f'{DRIVE_PROJECT}/runs',
        name=f'exp4_{name}',
        **aug_params,
    )

    train_time = time.time() - start

    best = YOLO(f'{DRIVE_PROJECT}/runs/exp4_{name}/weights/best.pt')
    val = best.val(data='/content/dataset/data.yaml', split='test')

    exp4_results[name] = extract_results(val, CLASS_NAMES)
    exp4_results[name]['training_time_min'] = round(train_time / 60, 1)

    print_hard_classes(exp4_results[name], f"Augmentation: {name}")

# Save results
with open(f'{DRIVE_PROJECT}/exp4_augmentation.json', 'w') as f:
    json.dump(exp4_results, f, indent=2)

print("\n\nEXPERIMENT 4 COMPLETE — Results saved!")
print("\nAugmentation comparison (overall mAP@0.5):")
for name, data in exp4_results.items():
    print(f"  {name}: {data['overall']['mAP50']}")

---
## Experiment 5: Transfer Learning vs From Scratch (RQ2)
**Question:** Does COCO pretraining help or hurt detection of visually challenging automotive components?

**Setup:** YOLOv8s pretrained (COCO) vs YOLOv8s from scratch

**Time estimate:** ~2 hours total (2 runs x ~60 min each)

In [ ]:
exp5_results = {}

configs = {
    'pretrained': 'yolov8s.pt',       # COCO pretrained weights
    'from_scratch': 'yolov8s.yaml',   # architecture only, random weights
}

for name, model_path in configs.items():
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"{'='*60}")

    model = YOLO(model_path)
    start = time.time()

    model.train(
        **COMMON_ARGS,
        imgsz=640,
        batch=16,
        project=f'{DRIVE_PROJECT}/runs',
        name=f'exp5_{name}',
    )

    train_time = time.time() - start

    best = YOLO(f'{DRIVE_PROJECT}/runs/exp5_{name}/weights/best.pt')
    val = best.val(data='/content/dataset/data.yaml', split='test')

    exp5_results[name] = extract_results(val, CLASS_NAMES)
    exp5_results[name]['training_time_min'] = round(train_time / 60, 1)

    print_hard_classes(exp5_results[name], f"Transfer Learning: {name}")
    print(f"Training time: {train_time/60:.1f} min")

# Save results
with open(f'{DRIVE_PROJECT}/exp5_transfer_learning.json', 'w') as f:
    json.dump(exp5_results, f, indent=2)

print("\n\nEXPERIMENT 5 COMPLETE — Results saved!")
print("\nTransfer learning comparison:")
for name, data in exp5_results.items():
    print(f"  {name}: mAP@0.5={data['overall']['mAP50']}, Time={data['training_time_min']}min")

---
## Experiment 6 (BONUS): YOLO26 vs YOLOv8 Comparison
**Question:** Does the latest YOLO generation (with STAL for small objects) improve detection of hard classes?

**Setup:** YOLO26s vs YOLOv8s (your baseline), same settings

**Time estimate:** ~1 hour

**Why this matters:** YOLO26 was released Sep 2025 — very few studies have tested it on domain-specific data. This makes your dissertation more novel.

In [ ]:
print("Training YOLO26s on car parts dataset...")
print("This uses the same settings as your YOLOv8s baseline for fair comparison.\n")

model_26 = YOLO('yolo26s.pt')
start = time.time()

model_26.train(
    **COMMON_ARGS,
    imgsz=640,
    batch=16,
    project=f'{DRIVE_PROJECT}/runs',
    name='exp6_yolo26s',
)

train_time_26 = time.time() - start

# Evaluate
best_26 = YOLO(f'{DRIVE_PROJECT}/runs/exp6_yolo26s/weights/best.pt')
val_26 = best_26.val(data='/content/dataset/data.yaml', split='test')

exp6_results = {
    'yolo26s': extract_results(val_26, CLASS_NAMES),
}
exp6_results['yolo26s']['training_time_min'] = round(train_time_26 / 60, 1)

# Save
with open(f'{DRIVE_PROJECT}/exp6_yolo26.json', 'w') as f:
    json.dump(exp6_results, f, indent=2)

print_hard_classes(exp6_results['yolo26s'], "YOLO26s")
print(f"\nTraining time: {train_time_26/60:.1f} min")
print("\nEXPERIMENT 6 COMPLETE — Results saved!")

---
## Results Analysis — Cross-Experiment Comparison
Run this after ALL experiments are complete to generate comparison tables and charts.

In [ ]:
# Load all results
all_results = {}
for f_path in sorted(glob.glob(f'{DRIVE_PROJECT}/exp*.json')):
    exp_name = os.path.basename(f_path).replace('.json', '')
    with open(f_path, 'r') as f:
        all_results[exp_name] = json.load(f)

print(f"Loaded {len(all_results)} experiment result files:")
for name in all_results:
    print(f"  - {name}")

In [ ]:
# Overall comparison table
print("\n" + "="*80)
print("OVERALL RESULTS COMPARISON")
print("="*80)
print(f"{'Experiment':<35} {'mAP50':>8} {'mAP50-95':>10} {'Precision':>10} {'Recall':>8}")
print("-"*75)

for exp_name, exp_data in all_results.items():
    if isinstance(exp_data, dict) and 'overall' in exp_data:
        # Single experiment result
        d = exp_data['overall']
        print(f"{exp_name:<35} {d['mAP50']:>8} {d['mAP50_95']:>10} {d['precision']:>10} {d['recall']:>8}")
    else:
        # Multi-variant experiment (e.g., resolution, augmentation)
        for variant, vdata in exp_data.items():
            if isinstance(vdata, dict) and 'overall' in vdata:
                d = vdata['overall']
                label = f"{exp_name}/{variant}"
                print(f"{label:<35} {d['mAP50']:>8} {d['mAP50_95']:>10} {d['precision']:>10} {d['recall']:>8}")

In [ ]:
# Hard classes comparison across ALL experiments
print("\n" + "="*80)
print("HARD CLASSES — mAP@0.5 ACROSS ALL EXPERIMENTS")
print("="*80)

hard_class_data = {}

for exp_name, exp_data in all_results.items():
    if isinstance(exp_data, dict) and 'per_class' in exp_data:
        for cls in HARD_CLASSES:
            if cls in exp_data['per_class']:
                if cls not in hard_class_data:
                    hard_class_data[cls] = {}
                hard_class_data[cls][exp_name] = exp_data['per_class'][cls]['mAP50']
    else:
        for variant, vdata in exp_data.items():
            if isinstance(vdata, dict) and 'per_class' in vdata:
                for cls in HARD_CLASSES:
                    if cls in vdata['per_class']:
                        if cls not in hard_class_data:
                            hard_class_data[cls] = {}
                        hard_class_data[cls][f"{exp_name}/{variant}"] = vdata['per_class'][cls]['mAP50']

# Print as table
if hard_class_data:
    df_hard = pd.DataFrame(hard_class_data).T
    print(df_hard.to_string())
    df_hard.to_csv(f'{DRIVE_PROJECT}/hard_classes_comparison.csv')
    print(f"\nSaved to {DRIVE_PROJECT}/hard_classes_comparison.csv")

In [ ]:
# Visualisation: Hard classes improvement chart
if hard_class_data:
    fig, axes = plt.subplots(1, len(HARD_CLASSES), figsize=(20, 5))

    for i, cls in enumerate(HARD_CLASSES):
        if cls in hard_class_data:
            experiments = list(hard_class_data[cls].keys())
            values = list(hard_class_data[cls].values())

            colors = ['#E24B4A' if v < 0.85 else '#EF9F27' if v < 0.92 else '#97C459' for v in values]
            axes[i].barh(range(len(experiments)), values, color=colors)
            axes[i].set_yticks(range(len(experiments)))
            axes[i].set_yticklabels([e.split('/')[-1] if '/' in e else e for e in experiments], fontsize=7)
            axes[i].set_xlim(0, 1)
            axes[i].set_title(cls, fontsize=9)
            axes[i].axvline(x=0.85, color='red', linestyle='--', linewidth=0.5, alpha=0.5)

    plt.suptitle('Hard Classes: mAP@0.5 Across All Experiments', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'{DRIVE_PROJECT}/hard_classes_chart.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Chart saved to {DRIVE_PROJECT}/hard_classes_chart.png")

In [ ]:
# Detection visualisations — save sample predictions for dissertation
import cv2

# Use best overall model for visualisation
best_model_path = f'{DRIVE_PROJECT}/runs/exp6_yolo26s/weights/best.pt'
if not os.path.exists(best_model_path):
    # Fallback to baseline if YOLO26 hasn't been run
    best_model_path = f'{DRIVE_PROJECT}/runs/exp1_yolov8s_baseline/weights/best.pt'

if os.path.exists(best_model_path):
    vis_model = YOLO(best_model_path)
    test_images = glob.glob('/content/dataset/test/images/*')[:12]

    results = vis_model.predict(
        source=test_images,
        conf=0.25,
        save=True,
        project=f'{DRIVE_PROJECT}/visualisations',
        name='sample_detections',
        line_width=2,
    )

    # Display
    fig, axes = plt.subplots(3, 4, figsize=(20, 15))
    for ax, r in zip(axes.flat, results):
        img = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'{len(r.boxes)} detections', fontsize=9)

    plt.suptitle('Sample Detection Results', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'{DRIVE_PROJECT}/sample_detections.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Visualisations saved!")
else:
    print(f"Model not found at {best_model_path}")
    print("Run experiments first, then come back to this cell.")

In [ ]:
# ONNX Export — for prototype demonstration in dissertation
if os.path.exists(best_model_path):
    export_model = YOLO(best_model_path)
    onnx_path = export_model.export(format='onnx', imgsz=640)
    print(f"\nModel exported to ONNX: {onnx_path}")
    print("This demonstrates deployment readiness — mention in Ch4 and Ch6.")
else:
    print("Run experiments first.")

In [ ]:
# Final summary — copy these numbers into your dissertation
print("\n" + "="*80)
print("COMPLETE EXPERIMENT SUMMARY FOR DISSERTATION")
print("="*80)

for exp_name, exp_data in all_results.items():
    print(f"\n--- {exp_name} ---")
    if isinstance(exp_data, dict) and 'overall' in exp_data:
        print(f"  Overall: {exp_data['overall']}")
        if 'training_time_min' in exp_data:
            print(f"  Training time: {exp_data['training_time_min']} min")
    else:
        for variant, vdata in exp_data.items():
            if isinstance(vdata, dict) and 'overall' in vdata:
                print(f"  {variant}: {vdata['overall']}")
                if 'training_time_min' in vdata:
                    print(f"    Time: {vdata['training_time_min']} min")

print("\n" + "="*80)
print("All results saved to Google Drive:")
print(f"  {DRIVE_PROJECT}/")
print("\nFiles generated:")
for f in sorted(glob.glob(f'{DRIVE_PROJECT}/*.*')):
    print(f"  {os.path.basename(f)}")